In [ ]:
# 1. Install Roboflow
!pip install roboflow ultralytics

# 2. Download your merged dataset
from roboflow import Roboflow

# PASTE YOUR ROBOFLOW EXPORT CODE HERE
from roboflow import Roboflow
rf = Roboflow(api_key="w8ORa2DZ62BYZmrQIqEw")
project = rf.workspace("thesis-meyy8").project("plate-recognition-kwrsk-z7k8h")
version = project.version(4)
dataset = version.download("yolo26")
# 3. Verify the Merge
import yaml

# Load the data.yaml file to check classes
with open(f"{dataset.location}/data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

print("Classes detected:", data_config['names'])
print("Number of classes:", data_config['nc'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 130.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...
Exporting format yolo26 in progress : 95.0%
Version export complete for yolo26 format



Extracting Dataset Version Zip to Plate-Recognition-2 in yolo26:: 100%|██████████| 7537/7537 [00:01<00:00, 5724.69it/s]

Classes detected: ['car', 'large vehicle', 'motorcycle', 'plate']
Number of classes: 4


In [ ]:
if data_config['nc'] != 4:
    print("⚠️ CRITICAL WARNING: Your dataset does not have 4 class. Check your Roboflow merge.")
else:
    print("✅ SUCCESS: Dataset ready for training.")

✅ SUCCESS: Dataset ready for training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO('yolo26n.pt')

# Train with Google Drive path
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=200,
    patience=20,
    imgsz=640,
    batch=-1,
    cache=True,
    fliplr=0.5,
    flipud=0.5,
    optimizer='auto',

    # --- CRITICAL CHANGE: SAVE TO GOOGLE DRIVE ---
    project='/content/drive/MyDrive/parking_yolo',  # <--- Points to your Drive
    name='parking_yolo26n_model',

    augment=True,
    cos_lr=True,
    save=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Plate-Recognition-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO('yolo26n.pt')

# Command to RESUME training from a crash
model = YOLO('/content/drive/MyDrive/parking_yolo/parking_yolo26n_model/weights/last.pt')
model.train(resume=True)

In [ ]:
from ultralytics import YOLO

# 1. Load the BEST weights from your previous run
# Based on your previous 'project' and 'name' settings:
path_to_best_weights = '/content/drive/MyDrive/parking_yolo/parking_yolo26n_model/weights/best.pt'
model = YOLO(path_to_best_weights)

# 2. Train again (Fine-tuning)
results = model.train(
    data=f"{dataset.location}/data.yaml",

    # Set how many *additional* epochs you want
    epochs=50,

    patience=20,
    imgsz=640,
    batch=-1,
    cache=True,
    fliplr=0.5,
    optimizer='auto',

    # Keep the same project folder
    project='/content/drive/MyDrive/parking_yolo',

    # 3. IMPORTANT: Change the name to avoid overwriting the old model
    name='parking_yolo26n_model_finetuned',

    augment=True,
    cos_lr=True,
    save=True
)

In [ ]:
from ultralytics import YOLO

# 1. Load the best weights saved in Google Drive
# We use the specific path defined in your previous training config
best_model = YOLO('/content/drive/MyDrive/parking_yolo/parking_yolo26n_model/weights/best.pt')

# 2. Validate it to get the final official numbers
print("--- FINAL VALIDATION ON TEST SET ---")
metrics = best_model.val(split='test', conf=0.25)

print(f"Final mAP50: {metrics.box.map50}")
print(f"Final mAP50-95: {metrics.box.map}")

In [ ]:
from IPython.display import Image, display
import os

# Define the path to your training run in Google Drive
run_path = '/content/drive/MyDrive/parking_yolo/parking_yolo26n_model'

# Path to the results graph
results_path = os.path.join(run_path, 'results.png')

print("--- TRAINING METRICS (Loss & mAP) ---")
if os.path.exists(results_path):
    display(Image(filename=results_path, width=800))
else:
    print(f"File not found at {results_path}. Check if your Google Drive is mounted.")

In [ ]:
# Path to the confusion matrix
conf_matrix_path = os.path.join(run_path, 'confusion_matrix.png')

# Sometimes YOLOv11 saves it as 'confusion_matrix_normalized.png'
conf_matrix_norm_path = os.path.join(run_path, 'confusion_matrix_normalized.png')

print("--- CONFUSION MATRIX ---")
if os.path.exists(conf_matrix_path):
    display(Image(filename=conf_matrix_path, width=800))
elif os.path.exists(conf_matrix_norm_path):
    display(Image(filename=conf_matrix_norm_path, width=800))
else:
    print("Confusion matrix not found. It might not have generated if training was stopped too early.")

In [ ]:
import glob

# Get the prediction images saved in the Drive folder
# These are generated during the validation phase
pred_images = glob.glob(f'{run_path}/val_batch*_pred.jpg')

print(f"--- SAMPLE PREDICTIONS ({len(pred_images)} batches found) ---")

# Display up to 3 batches to visually check detection quality
for img_path in pred_images[:3]:
    print(f"Displaying: {os.path.basename(img_path)}")
    display(Image(filename=img_path, width=900))

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# 1. Load your BEST fine-tuned model
# Adjust path if needed
model_path = '/content/drive/MyDrive/parking_yolo/parking_yolo26n_model/weights/best.pt'
model = YOLO(model_path)

# 2. Run Validation on the TEST split
# This automatically generates confusion_matrix.png
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split='test',       # <--- Specifies using the "test" images from your yaml
    project='/content/drive/MyDrive/parking_yolo',
    name='test_evaluation',
    conf=0.25,          # Confidence threshold (adjust as needed)
    iou=0.6,            # NMS IoU threshold
    plots=True          # Ensures confusion matrix is generated
)

# 3. Display the Confusion Matrix directly in the notebook
# The file is saved in: project/name/confusion_matrix.png
cm_path = '/content/drive/MyDrive/parking_yolo/test_evaluation/confusion_matrix.png'

if os.path.exists(cm_path):
    img = mpimg.imread(cm_path)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Confusion Matrix (Test Set)")
    plt.show()
else:
    print("Confusion matrix not found. Check if the validation run completed successfully.")

# Optional: Print specific class metrics
print(f"Map50: {metrics.box.map50}")
print(f"Map50-95: {metrics.box.map}")